# Integração de Pydantic com FastAPI

Este notebook demonstra como utilizar o **Pydantic** em conjunto com o **FastAPI** para criar uma API robusta, com validação automática de dados, serialização de UUIDs e testes automatizados utilizando o `TestClient`.

## 1. Importações

Importamos as ferramentas do FastAPI para a criação da API e do Pydantic para a modelagem dos dados.

In [1]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

## 2. Definição do Modelo User

O modelo `User` utiliza várias funcionalidades avançadas:
- **`model_config`**: Proíbe campos extras (`extra: forbid`).
- **`default_factory`**: Gera valores dinâmicos como UUIDs e Timestamps.
- **`kw_only=True`**: Garante que certos campos sejam passados apenas por nome.
- **`field_serializer`**: Garante que o UUID seja convertido para string no JSON.

In [2]:
app = FastAPI()

class User(BaseModel):
    model_config = {
        "extra": "forbid",
    }
    __users__ = []
    
    name: str = Field(..., description="Name of the user")
    email: EmailStr = Field(..., description="Email address of the user")
    friends: list[UUID4] = Field(
        default_factory=list, max_length=500, description="List of friends"
    )
    blocked: list[UUID4] = Field(
        default_factory=list, max_length=500, description="List of blocked users"
    )
    signup_ts: Optional[datetime] = Field(
        default_factory=datetime.now, description="Signup timestamp", kw_only=True
    )
    id: UUID4 = Field(
        default_factory=uuid4, description="Unique identifier", kw_only=True
    )

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)

## 3. Endpoints da API

Definimos rotas para listar, criar e obter utilizadores específicos. O FastAPI utiliza o `response_model` para filtrar e validar a saída.

In [3]:
@app.get("/users", response_model=list[User])
async def get_users() -> list[User]:
    return list(User.__users__)

@app.post("/users", response_model=User)
async def create_user(user: User):
    User.__users__.append(user)
    return user

@app.get("/users/{user_id}", response_model=User)
async def get_user(user_id: UUID4) -> User | JSONResponse:
    try:
        return next((user for user in User.__users__ if user.id == user_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "User not found"})

## 4. Testes Automatizados

Utilizamos o `TestClient` para simular requisições à API e validar se o comportamento está de acordo com o esperado.

In [4]:
with TestClient(app) as client:
    # Criar utilizadores
    for i in range(5):
        response = client.post(
            "/users",
            json={"name": f"User {i}", "email": f"example{i}@arjancodes.com"},
        )
        assert response.status_code == 200

    # Listar utilizadores
    response = client.get("/users")
    assert response.status_code == 200
    assert len(response.json()) == 5

    # Testar erro de validação (email inválido)
    response = client.post("/users", json={"name": "User 6", "email": "wrong"})
    assert response.status_code == 422
    
    print("All tests passed successfully!")

All tests passed successfully!
